In [1]:
import io
import os
import pandas as pd
import paramiko
import matplotlib.pyplot as plt
# --- 提供いただいたヘッダー見本（タブ区切りテキスト） ---
raw_header_text = """Timestamp	Position	Temperature	Wiper Count	40m CNR (dB)	40m Radial Wind Speed (m/s)	40m Radial Wind Speed Dispersion (m/s)	40m Wind Speed (m/s)	40m Wind Direction (ｰ)	40m X-wind (m/s)	40m Y-wind (m/s)	40m Z-wind (m/s)		60m CNR (dB)	60m Radial Wind Speed (m/s)	60m Radial Wind Speed Dispersion (m/s)	60m Wind Speed (m/s)	60m Wind Direction (ｰ)	60m X-wind (m/s)	60m Y-wind (m/s)	60m Z-wind (m/s)		80m CNR (dB)	80m Radial Wind Speed (m/s)	80m Radial Wind Speed Dispersion (m/s)	80m Wind Speed (m/s)	80m Wind Direction (ｰ)	80m X-wind (m/s)	80m Y-wind (m/s)	80m Z-wind (m/s)		100m CNR (dB)	100m Radial Wind Speed (m/s)	100m Radial Wind Speed Dispersion (m/s)	100m Wind Speed (m/s)	100m Wind Direction (ｰ)	100m X-wind (m/s)	100m Y-wind (m/s)	100m Z-wind (m/s)		120m CNR (dB)	120m Radial Wind Speed (m/s)	120m Radial Wind Speed Dispersion (m/s)	120m Wind Speed (m/s)	120m Wind Direction (ｰ)	120m X-wind (m/s)	120m Y-wind (m/s)	120m Z-wind (m/s)		140m CNR (dB)	140m Radial Wind Speed (m/s)	140m Radial Wind Speed Dispersion (m/s)	140m Wind Speed (m/s)	140m Wind Direction (ｰ)	140m X-wind (m/s)	140m Y-wind (m/s)	140m Z-wind (m/s)		160m CNR (dB)	160m Radial Wind Speed (m/s)	160m Radial Wind Speed Dispersion (m/s)	160m Wind Speed (m/s)	160m Wind Direction (ｰ)	160m X-wind (m/s)	160m Y-wind (m/s)	160m Z-wind (m/s)		180m CNR (dB)	180m Radial Wind Speed (m/s)	180m Radial Wind Speed Dispersion (m/s)	180m Wind Speed (m/s)	180m Wind Direction (ｰ)	180m X-wind (m/s)	180m Y-wind (m/s)	180m Z-wind (m/s)		200m CNR (dB)	200m Radial Wind Speed (m/s)	200m Radial Wind Speed Dispersion (m/s)	200m Wind Speed (m/s)	200m Wind Direction (ｰ)	200m X-wind (m/s)	200m Y-wind (m/s)	200m Z-wind (m/s)		220m CNR (dB)	220m Radial Wind Speed (m/s)	220m Radial Wind Speed Dispersion (m/s)	220m Wind Speed (m/s)	220m Wind Direction (ｰ)	220m X-wind (m/s)	220m Y-wind (m/s)	220m Z-wind (m/s)		240m CNR (dB)	240m Radial Wind Speed (m/s)	240m Radial Wind Speed Dispersion (m/s)	240m Wind Speed (m/s)	240m Wind Direction (ｰ)	240m X-wind (m/s)	240m Y-wind (m/s)	240m Z-wind (m/s)		260m CNR (dB)	260m Radial Wind Speed (m/s)	260m Radial Wind Speed Dispersion (m/s)	260m Wind Speed (m/s)	260m Wind Direction (ｰ)	260m X-wind (m/s)	260m Y-wind (m/s)	260m Z-wind (m/s)		280m CNR (dB)	280m Radial Wind Speed (m/s)	280m Radial Wind Speed Dispersion (m/s)	280m Wind Speed (m/s)	280m Wind Direction (ｰ)	280m X-wind (m/s)	280m Y-wind (m/s)	280m Z-wind (m/s)		300m CNR (dB)	300m Radial Wind Speed (m/s)	300m Radial Wind Speed Dispersion (m/s)	300m Wind Speed (m/s)	300m Wind Direction (ｰ)	300m X-wind (m/s)	300m Y-wind (m/s)	300m Z-wind (m/s)"""
clean_headers = [c.strip() for c in raw_header_text.split("\t") if c.strip() != ""]

# --- SFTP接続設定 ---
sftp_host = "172.16.34.223"
sftp_user = "hayashi"
sftp_pass = "hayashi"
remote_dir = "/home/hayashi/kaiseki/SampleData/FREA_LiDAR/WLS71045_rtd_files/"

# 1. ここで確実にSSHとSFTPの接続を行います
ssh = paramiko.SSHClient()
ssh.set_missing_host_key_policy(paramiko.AutoAddPolicy())
ssh.connect(sftp_host, username=sftp_user, password=sftp_pass)
sftp = ssh.open_sftp()  # ここで 'sftp' が定義されます

df_list = []

try:
    # 2. 接続が確立された状態でループを回します
    for n in range(1, 29):
        date_str = f"{n:02d}"
        file_name = f"WLS71045_2025_02_{date_str}__00_00_00.rtd"
        remote_path = os.path.join(remote_dir, file_name)

        print(f"ファイルを処理中: {file_name}")

        try:
            with sftp.open(remote_path, "rb") as f:
                binary_data = f.read()

            text_data = binary_data.decode("shift_jis", errors="ignore")

            df_raw = pd.read_csv(
                io.StringIO(text_data),
                sep="\s+",
                skiprows=42,
                header=None,
            )

            if df_raw.empty:
                print(f"⚠️ {file_name} はデータが空のためスキップします。")
                continue

            # 日時の結合
            combined_datetime = pd.to_datetime(
                df_raw[0].astype(str) + " " + df_raw[1].astype(str),
                errors="coerce",
            )

            df_clean = pd.DataFrame()
            df_clean[0] = combined_datetime

            # 残りのデータ列を結合
            df_data_part = df_raw.iloc[:, 2:]
            df_clean = pd.concat([df_clean, df_data_part], axis=1)
            df_clean.columns = range(df_clean.shape[1])

            # 列名のマッピング
            final_columns = []
            actual_cols = df_clean.shape[1]
            for i in range(actual_cols):
                if i < len(clean_headers):
                    final_columns.append(clean_headers[i])
                else:
                    final_columns.append(f"Extra_Column_{i}")
            df_clean.columns = final_columns

            df_list.append(df_clean)

        except FileNotFoundError:
            print(
                f"⚠️ ファイルが存在しません（スキップします）: {file_name}"
            )
        except Exception as file_error:
            print(f"❌ ファイル {file_name} の処理中にエラー: {file_error}")

    # すべてのファイルを縦結合
    if df_list:
        print("\n全ファイルを1つのDataFrameに統合中...")
        df_rtd = pd.concat(df_list, axis=0, ignore_index=True)

        print("\n🎉 全ファイルの統合・整形が完全に成功しました！")
        print("最終データ形状（合計行数, 列数）:", df_rtd.shape)
        print("\n--- データの最初の3行 ---")
        print(df_rtd.head(3))
    else:
        print("\n⚠️ 有効なデータが1つも見つかりませんでした。")

except Exception as e:
    print(f"\n❌ エラーが発生しました: {e}")

finally:
    # 3. 最後に安全にクローズします
    sftp.close()
    ssh.close()

ファイルを処理中: WLS71045_2025_02_01__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_02__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_03__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_04__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_05__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_06__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_07__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_08__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_09__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_10__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_11__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_12__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_13__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_14__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_15__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_16__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_17__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_18__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_19__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_20__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_21__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_22__00_00_00.rtd
ファイルを処理中: WLS71045_2025_02_23__0

In [2]:
df_rtd.to_pickle("Sample_df_rtd.pkl")

In [3]:
df_rtd

,Timestamp,Position,Temperature,Wiper Count,40m CNR (dB),40m Radial Wind Speed (m/s),40m Radial Wind Speed Dispersion (m/s),40m Wind Speed (m/s),40m Wind Direction (ｰ),40m X-wind (m/s),...,280m Y-wind (m/s),280m Z-wind (m/s),300m CNR (dB),300m Radial Wind Speed (m/s),300m Radial Wind Speed Dispersion (m/s),300m Wind Speed (m/s),300m Wind Direction (ｰ),300m X-wind (m/s),300m Y-wind (m/s),300m Z-wind (m/s)
0,2025-02-01 00:00:00.680,90,0.5,0,-8.04,0.40,0.02,3.04,170.28,-2.992,...,-3.689,0.386,-13.37,-1.11,0.02,4.41,233.32,-2.637,-3.541,0.258
1,2025-02-01 00:00:01.480,180,0.5,0,-10.50,2.03,0.02,2.98,170.10,-2.937,...,-3.689,0.386,-10.97,0.31,0.02,4.03,241.43,-1.928,-3.541,0.258
2,2025-02-01 00:00:02.280,270,0.5,0,-9.11,-0.08,0.02,2.98,170.09,-2.937,...,-3.611,0.386,-11.86,2.20,0.02,4.02,241.32,-1.928,-3.525,0.258
3,2025-02-01 00:00:03.080,V,0.5,0,-3.68,0.24,0.02,NaN,NaN,NaN,...,NaN,0.562,-6.26,0.53,0.02,NaN,NaN,NaN,NaN,0.527
4,2025-02-01 00:00:03.880,0,0.5,0,-8.07,-1.02,0.02,3.29,171.03,-3.253,...,-3.611,0.562,-12.02,-1.34,0.02,3.94,243.43,-1.763,-3.525,0.527
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3007742,2025-02-28 23:59:56.240,0,164.0,0,-14.74,-2.17,0.02,4.39,169.39,-4.313,...,-3.228,-1.254,-18.79,-2.58,0.02,6.73,220.14,-5.148,-4.341,0.338
3007743,2025-02-28 23:59:57.040,90,164.0,0,-15.31,0.04,0.02,4.36,171.82,-4.313,...,-3.087,-1.254,-19.81,-1.81,0.02,6.64,219.19,-5.148,-4.198,0.338
3007744,2025-02-28 23:59:57.840,180,164.0,0,-15.12,2.20,0.02,4.70,172.42,-4.658,...,-3.087,-1.254,-19.83,1.70,0.02,6.19,222.69,-4.551,-4.198,0.338
3007745,2025-02-28 23:59:58.640,270,164.0,0,-15.13,-0.73,0.02,4.73,170.01,-4.658,...,-4.353,-1.254,-19.41,1.88,0.02,6.02,220.85,-4.551,-3.935,0.338
